<a href="https://colab.research.google.com/github/Ken89MathCompSci/BERT4Nilm-base/blob/kengoh-corrected-parameters-fridge-and-washer/22-September-2025-Rerunning-fridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
ls

BERT4Nilm-base/  sample_data/


In [ ]:
pwd

'/content'

In [ ]:
import fileinput

trainer_file = '/content/BERT4Nilm-base/trainer.py'

# Use a more robust in-place replacement with python
with fileinput.FileInput(trainer_file, inplace=True, backup='.bak') as file:
    for line in file:
        # Remove the deprecated import
        if 'from torch.autograd.gradcheck import zero_gradients' in line:
            continue # Skip printing this line
        # Replace the deprecated function call with the optimizer's zero_grad() method
        elif 'zero_gradients(self.model.cpu().parameters())' in line:
            # It's important to preserve indentation
            indentation = line[:len(line) - len(line.lstrip())]
            print(f"{indentation}self.optim.zero_grad()");
        else:
            print(line, end='')

print(f"Patched {trainer_file} to fix the 'zero_gradients' ImportError.")

Patched /content/BERT4Nilm-base/trainer.py to fix the 'zero_gradients' ImportError.


In [ ]:
with open('/content/BERT4Nilm-base/dataset.py', 'r') as f:
    content = f.read()

# Correct the deprecated pandas '.append' method
# This handles all the broken states we've seen so far
content = content.replace('_entire_data._append', 'entire_data._append') # As seen in the last error
content = content.replace('_entire_data.append', 'entire_data._append') # From the first failed fix
content = content.replace('entire_data.append', 'entire_data._append')  # Original state

# Correct the deprecated '.fillna' method
content = content.replace(".fillna(method='ffill')", ".ffill()")

with open('/content/BERT4Nilm-base/dataset.py', 'w') as f:
    f.write(content)

print("dataset.py has been patched (again, with feeling).")

dataset.py has been patched (again, with feeling).


In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU is available.")
    # List all available GPUs and their IDs
    for i in range(torch.cuda.device_count()):
        print(f"GPU ID: {i}, Name: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU available. The code will run on CPU.")

No GPU available. The code will run on CPU.


In [ ]:
!cd /content/BERT4Nilm-base && python train.py

Input r for REDD, u for UK_DALE: r
Input r, w, m or d for target appliance: r
Input training epochs: 100
Appliance: ['refrigerator']
Sum of ons: [49487.]
Total length: 959479
C0: 9.999999974752427e-07
Failed to load old model, continue training new model...
  0% 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Validation, rel_err 0.00, acc 1.00, f1 0.00: 100% 2/2 [00:11<00:00,  5.59s/it]
Saving best model...
  0% 0/57 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Epoch 1, loss 0.47: 100% 57/57 [11:24<00:00, 12.00s/it]
Validation, rel_err 0.07, acc 0.93, f1 0.01: 100% 2/2 [00:06<00:00,  3.47s/it]
  0% 0/57 [0

In [ ]:
!cd /content/BERT4Nilm-base && \
  sed -i "/parser = argparse.ArgumentParser(/a\    parser.add_argument('--sampling', type=int, default=5, help='Sampling rate')" debug_predictions.py && \
  python debug_predictions.py

Appliance: ['refrigerator']
Sum of ons: [66159.]
Total length: 263308
Dataset loaded with 263308 samples
Appliance names: ['refrigerator']
Thresholds: [10]
Min on: [5]
Min off: [2]

Ground truth analysis:
refrigerator: 66159/263308 samples ON (25.13%)
  Power when ON: mean=198.0W, max=400.0W, min=6.0W
Model not found at experiments/redd_lf/refrigerator/best_acc_model.pth
